# Farm, Cat, and Economic Data Merge

This notebook extends the previously merged farm and cat dataset by adding economic cost data.

The economic dataset contains annual cost estimates related to invasive species impacts, including information on total costs, cat-only costs, costs involving cats together with other species, as well as breakdowns by impacted sector and type of cost.

The goal of this notebook is to:

1. load the cleaned economic dataset and the merged farm-cat dataset;
2. harmonize territory names for merging;
3. construct yearly economic indicators by territory;
4. create additional variables for cat-only and mixed-species costs;
5. aggregate costs by impacted sector and type of cost;
6. merge the economic data with farm and cat variables;
7. save the final result as `farm_cat_eco.csv`.

The resulting dataset will later be extended with news variables for the final stage of the project.

In [1]:
import pandas as pd
import re

eco_df = pd.read_csv("../datasets/eco_df_clean.csv")
farm_cat_merged = pd.read_csv("../datasets/farm_cat.csv")

In [2]:
# Inspect territory naming before merging
eco_df["State|Province|Administrative_area"].unique()

array(['Unspecified', 'Western Australia', 'Perth', 'Australia',
       'South Australia', 'New South Wales', 'Victoria', 'Queensland',
       'Tasmania'], dtype=object)

In [3]:
farm_cat_merged["territory"].unique()

array(['All Australia', 'New South Wales', 'Northern Territory',
       'Queensland', 'South Australia', 'Tasmania', 'Victoria',
       'Western Australia'], dtype=object)

In [ ]:
# Prepare territory names for merging
eco_df["territory"] = eco_df["State|Province|Administrative_area"].astype(str).str.strip()

eco_df["territory"] = eco_df["territory"].replace({
    "Perth": "Western Australia", # convert city name to state
    "Australia": "Unspecified", # treat national-level entry as unspecified
})

eco_df = eco_df.drop(["State|Province|Administrative_area"], axis=1)

In [7]:
# Define the main cost variable used throughout the notebook
COST = "Cost_estimate_per_year_2017_USD_exchange_rate"

In [9]:
# Separate cat-only costs from costs involving cats plus other species
eco_cat_only = (
    eco_df["Common_name"].astype(str).str.lower().str.contains(r"\bcat\b", na=False)
    & ~eco_df["Common_name"].astype(str).str.contains(r",|\+|/|;|\band\b", case=False, na=False)
)

eco_df["cost_cat_only"] = eco_df[COST].where(eco_cat_only, 0)
eco_df["cost_cat_plus_other"] = eco_df[COST].where(~eco_cat_only, 0)

In [10]:
# Add national-level observations by duplicating all rows and assigning territory = "All Australia"
eco_all = eco_df.copy()
eco_all["territory"] = "All Australia"

eco_df2 = pd.concat([eco_df, eco_all], ignore_index=True)

In [13]:
# Define grouping keys
keys = ["Publication_year", "territory"]

# Helper function for clean column names
def slug(x):
    x = str(x).strip()
    x = re.sub(r"\s+", "_", x)
    x = re.sub(r"[^0-9a-zA-Z_]+", "_", x)
    return x.strip("_").lower()

# Aggregate total economic costs by year and territory
eco_totals = (
    eco_df2.groupby(keys, as_index=False)
    .agg(
        eco_cost_total=(COST, "sum"),
        eco_cost_cat_only=("cost_cat_only", "sum"),
        eco_cost_cat_plus_other=("cost_cat_plus_other", "sum"),
    )
)

# Aggregate economic costs by impacted sector
sector = (
    eco_df2.pivot_table(
        index=keys,
        columns="Impacted_sector",
        values=COST,
        aggfunc="sum",
        fill_value=0
    )
)

sector.columns = [f"eco_sector_{slug(c)}" for c in sector.columns]
sector = sector.reset_index()

# Aggregate economic costs by type of cost
ctype = (
    eco_df2.pivot_table(
        index=keys,
        columns="Type_of_cost",
        values=COST,
        aggfunc="sum",
        fill_value=0
    )
)

ctype.columns = [f"eco_type_{slug(c)}" for c in ctype.columns]
ctype = ctype.reset_index()

# Build the final aggregated economic dataset
eco_year_territory = (
    eco_totals
    .merge(sector, on=keys, how="left")
    .merge(ctype, on=keys, how="left")
)

In [14]:
# Fill missing values in numeric columns with zero
num_cols = eco_year_territory.columns.difference(keys)
eco_year_territory[num_cols] = eco_year_territory[num_cols].fillna(0)

# Sort the dataset
eco_year_territory = eco_year_territory.sort_values(keys).reset_index(drop=True)

# Rename year column for consistency with the farm-cat dataset
eco_year_territory = eco_year_territory.rename(columns={"Publication_year": "year"})

In [15]:
# Merge farm-cat data with economic data
# Restrict farm-cat data to years covered by the cost dataset
keys = ["year", "territory"]

farm_cat_merged = farm_cat_merged[farm_cat_merged["year"] >= 2002]

farm_cat_eco_merged = farm_cat_merged.merge(
    eco_year_territory,
    on=keys,
    how="outer"
)

In [16]:
farm_cat_eco_merged

,year,territory,lambs,rams,ewes,lamb_sheep_shorn,sheep_flock,sheep_purchased,cats_occurrence_total,cats_forest,...,eco_sector_authorities_stakeholders,eco_sector_environment,eco_sector_health,eco_type_control,eco_type_control_education,eco_type_control_medical_care,eco_type_control_prevention,eco_type_damage_loss,eco_type_eradication,eco_type_prevention
0,2002,All Australia,10658.0,662.0,26531.0,51883.0,47861.0,3300.0,197.0,77.0,...,825276.359414,0.0,0.0,825276.359414,0.0,0.0,0.0,0.0,0.0,0.0
1,2002,New South Wales,2624.0,131.0,5996.0,11360.0,10544.0,770.0,104.0,46.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2002,Northern Territory,0.0,0.0,0.0,0.0,0.0,0.0,34.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2002,Queensland,1421.0,103.0,5020.0,11123.0,9707.0,584.0,24.0,17.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2002,South Australia,2148.0,137.0,5372.0,8857.0,8938.0,692.0,18.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174,2022,Queensland,1056.0,59.0,2039.0,3267.0,3706.0,153.0,78.0,16.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
175,2022,South Australia,3003.0,123.0,5280.0,9578.0,8842.0,452.0,150.0,14.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
176,2022,Tasmania,567.0,20.0,1029.0,1804.0,1795.0,55.0,118.0,45.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
177,2022,Victoria,1608.0,50.0,2272.0,4236.0,4239.0,548.0,183.0,116.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
farm_cat_eco_merged.to_csv("../datasets/farm_cat_eco.csv", index=False)